# Markov Models

In this lab we will use Markov models to study human languages.

## Data Preparation

Before we can actually think about mathematical modeling, we need to gather some sample data to work on. For this lab, we will be retrieving text from the website of [Project Gutenberg](https://www.gutenberg.org/).

Go to that project website and find the page for "The Hound of the Baskervilles" novel. In particular, load the simple text form of the book.

Now define a Python variable `novel` and assign to it the first few paragraphs of Chapter 1.

<div class="alert alert-info">
In Python there is a specific syntax to define multi-line strings. If you use such syntax, you can simply copy and paste the novel text from the Project Gutenberg website.
</div>

In [64]:
novel = """
      Mr. Sherlock Holmes, who was usually very late in the mornings,
      save upon those not infrequent occasions when he was up all
      night, was seated at the breakfast table. I stood upon the
      hearth-rug and picked up the stick which our visitor had left
      behind him the night before. It was a fine, thick piece of wood,
      bulbous-headed, of the sort which is known as a “Penang lawyer.”
      Just under the head was a broad silver band nearly an inch
      across. “To James Mortimer, M.R.C.S., from his friends of the
      C.C.H.,” was engraved upon it, with the date “1884.” It was just
      such a stick as the old-fashioned family practitioner used to
      carry—dignified, solid, and reassuring.

      “Well, Watson, what do you make of it?”

      Holmes was sitting with his back to me, and I had given him no
      sign of my occupation.

      “How did you know what I was doing? I believe you have eyes in
      the back of your head.”

      “I have, at least, a well-polished, silver-plated coffee-pot in
      front of me,” said he. “But, tell me, Watson, what do you make of
      our visitor’s stick? Since we have been so unfortunate as to miss
      him and have no notion of his errand, this accidental souvenir
      becomes of importance. Let me hear you reconstruct the man by an
      examination of it.”

      “I think,” said I, following as far as I could the methods of my
      companion, “that Dr. Mortimer is a successful, elderly medical
      man, well-esteemed since those who know him give him this mark of
      their appreciation.”

      “Good!” said Holmes. “Excellent!”

"""

The text that you have just copied it's too complex for our purposes. It contains not only words, but also punctuation marks and empty lines.

At the and of our data preparation we want a string containing only words separated by single spaces. Say we start from:

```
“I have, at least, a well-polished, silver-plated coffee-pot in front of
me,” said he.
```

Our final output should be:

```
i have at least a well polished silver pated coffee pot in front of me said he
```

Your first task is therefore to simplify the data, performing the following steps (not necessarily in this order):
* remove the distinction from upper and lower case, for instance converting everything to lower case;
* remove all characters that are not letters or spaces;
* remove empty lines and redundant spaces (words must be separated by a **single** space).

Remember that Python strings behave like containers. They are collections of single characters that you can iterate upon using `for` loops or list comprehensions.

You may find the following two functions useful for converting strings into actual lists and vice-versa.

In [18]:
def string2list(s):
    return list(s)

In [19]:
def list2string(l):
    return ''.join(l)  # join all characters together

And the following two functions for converting strings into list of words and vice-versa.

In [20]:
def string2words(s):
    return s.split(' ')  # split on spaces

In [51]:
def words2string(ws):
    return ' '.join(ws)  # join words with spaces between each pair

Now implement the transformations described above.

In [62]:
import string
import unicodedata

def clean_string(text):
    # convert text to lowercase, replace new lines, remove punctuations
    extra_punct = "“”‘’—–…"
    text = text.lower().replace("\n", " ").translate(str.maketrans('', '', string.punctuation + extra_punct))   

    final_text = words2string(text.split())
    
    return final_text
    

In [66]:
print(clean_string(novel))

mr sherlock holmes who was usually very late in the mornings save upon those not infrequent occasions when he was up all night was seated at the breakfast table i stood upon the hearthrug and picked up the stick which our visitor had left behind him the night before it was a fine thick piece of wood bulbousheaded of the sort which is known as a penang lawyer just under the head was a broad silver band nearly an inch across to james mortimer mrcs from his friends of the cch was engraved upon it with the date 1884 it was just such a stick as the oldfashioned family practitioner used to carrydignified solid and reassuring well watson what do you make of it holmes was sitting with his back to me and i had given him no sign of my occupation how did you know what i was doing i believe you have eyes in the back of your head i have at least a wellpolished silverplated coffeepot in front of me said he but tell me watson what do you make of our visitors stick since we have been so unfortunate as

In [67]:
novel_clean = clean_string(novel)

Now that we have cleaned up the text we will be using, there is one more conversion we have to make before we think abount the mathematical modeling.

Each character in our string will be represented as a different _state_ of a Markov model. As states are usually labeled with integer numbers (1, 2 ...), we need a way to convert each letter into an integer.

Start by defining a function which takes the list of all possible characters (which you have used above for the cleanup) and assign to each a different number (starting from 0). For instance, one possible pairing would be:

```
a -> 0
b -> 1
c -> 2
...
```

In [73]:
def letters2int(ls):
    counter = 0
    letter_int_mapping = {} 
    
    for i in ls:
        if i not in letter_int_mapping.keys() and i.isalpha():
            letter_int_mapping[i] =  counter
            counter += 1
    
    return letter_int_mapping
    

In [102]:
unique_dict = letters2int(novel_clean)

print(unique_dict)

{'m': 0, 'r': 1, 's': 2, 'h': 3, 'e': 4, 'l': 5, 'o': 6, 'c': 7, 'k': 8, 'w': 9, 'a': 10, 'u': 11, 'y': 12, 'v': 13, 't': 14, 'i': 15, 'n': 16, 'g': 17, 'p': 18, 'f': 19, 'q': 20, 'd': 21, 'b': 22, 'j': 23, 'x': 24}


Now write a second function that:
* converts the cleaned-up text into a list of characters;
* replaces each character with the correct number.

Using the mapping defined above, the string
```
acca
```

would become
```python
[0, 2, 2, 0]
```

In [103]:
def string2intsTEST(s):
    s_map = letters2int(s)
    s_list = []

    for char in s:
        if char.isalpha():
            char = s_map[char]
            s_list.append(char)

    return s_list

In [86]:
def string2ints(s):
    s_map = letters2int(s)
    s_list = [s_map[char] for char in s if char.isalpha()]
    return s_list

In [105]:
mapped_novel = string2ints(novel_clean)

print(mapped_novel)

[0, 1, 2, 3, 4, 1, 5, 6, 7, 8, 3, 6, 5, 0, 4, 2, 9, 3, 6, 9, 10, 2, 11, 2, 11, 10, 5, 5, 12, 13, 4, 1, 12, 5, 10, 14, 4, 15, 16, 14, 3, 4, 0, 6, 1, 16, 15, 16, 17, 2, 2, 10, 13, 4, 11, 18, 6, 16, 14, 3, 6, 2, 4, 16, 6, 14, 15, 16, 19, 1, 4, 20, 11, 4, 16, 14, 6, 7, 7, 10, 2, 15, 6, 16, 2, 9, 3, 4, 16, 3, 4, 9, 10, 2, 11, 18, 10, 5, 5, 16, 15, 17, 3, 14, 9, 10, 2, 2, 4, 10, 14, 4, 21, 10, 14, 14, 3, 4, 22, 1, 4, 10, 8, 19, 10, 2, 14, 14, 10, 22, 5, 4, 15, 2, 14, 6, 6, 21, 11, 18, 6, 16, 14, 3, 4, 3, 4, 10, 1, 14, 3, 1, 11, 17, 10, 16, 21, 18, 15, 7, 8, 4, 21, 11, 18, 14, 3, 4, 2, 14, 15, 7, 8, 9, 3, 15, 7, 3, 6, 11, 1, 13, 15, 2, 15, 14, 6, 1, 3, 10, 21, 5, 4, 19, 14, 22, 4, 3, 15, 16, 21, 3, 15, 0, 14, 3, 4, 16, 15, 17, 3, 14, 22, 4, 19, 6, 1, 4, 15, 14, 9, 10, 2, 10, 19, 15, 16, 4, 14, 3, 15, 7, 8, 18, 15, 4, 7, 4, 6, 19, 9, 6, 6, 21, 22, 11, 5, 22, 6, 11, 2, 3, 4, 10, 21, 4, 21, 6, 19, 14, 3, 4, 2, 6, 1, 14, 9, 3, 15, 7, 3, 15, 2, 8, 16, 6, 9, 16, 10, 2, 10, 18, 4, 16, 10, 16, 17, 5,

## Markov Model, order 1

Here we are interested in estimating an order 1 Markov model representing our text.

In practice, we need to build a matrix capturing all the transition probabilities between pairs of characters.

The actual workflow will be we following:
* build a numpy matrix representing transitions between characters of the appropriate size;
* the matrix should be initially filled with zeroes;
* for each pair of characters (say 'a' and 'b') count how many times the pair 'ab' appears in the text and store it in the matrix;
* transform the frequency matrix into a proability matrix.

Implement in Python code the above steps. You may want to start doing one step at a time, interacting with the notebook. Later you should gather all the code into a single function like the following:

```python
def estimate_markov_1(s):  # s is the input text
    # do stuff
    return m  # m is the computed probability matrix
```

In [88]:
import numpy as np

In [112]:
matrix = np.zeros((len(unique_dict), len(unique_dict)))
print(matrix)

[[0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
  0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
  0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
  0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
  0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
  0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
  0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
  0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
  0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
  0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
  0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
  0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
  0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.

In [113]:
for i in range(len(mapped_novel) - 1):
    curr_pos = mapped_novel[i]
    next_pos = mapped_novel[i + 1]

    row = curr_pos
    col = next_pos

    matrix[row, col] += 1


In [114]:
print(matrix)

[[ 0.  2.  0.  1. 14.  0.  3.  0.  0.  0.  6.  0.  2.  0.  2.  3.  1.  1.
   2.  0.  0.  0.  0.  0.  0.]
 [ 2.  2.  2.  2.  6.  3.  4.  1.  1.  0.  5.  3.  3.  2.  7.  3.  1.  0.
   1.  0.  0.  0.  2.  1.  0.]
 [ 2.  0.  8.  5.  7.  0.  9.  0.  1.  3.  9.  6.  0.  0. 13. 11.  0.  0.
   0.  4.  0.  1.  1.  1.  0.]
 [ 0.  1.  0.  1. 24.  0. 10.  0.  0.  1. 12.  0.  0.  0.  3. 17.  0.  0.
   0.  0.  0.  0.  0.  0.  0.]
 [ 4. 11. 11.  6.  4.  8.  4.  5.  0.  4. 13.  1.  2.  1.  4.  5. 13.  0.
   1.  2.  1. 12.  4.  0.  2.]
 [ 5.  0.  1.  0.  7.  8.  2.  0.  0.  1.  3.  0.  4.  2.  0.  3.  1.  0.
   1.  0.  0.  3.  1.  0.  0.]
 [ 5.  9.  4.  0.  0.  7.  3.  4.  1.  6.  1. 12.  2.  0.  3.  1. 16.  0.
   0. 14.  0.  4.  0.  1.  0.]
 [ 0.  1.  1.  5.  6.  0.  5.  5.  8.  0.  3.  1.  0.  0.  2.  2.  0.  0.
   0.  0.  0.  0.  0.  0.  0.]
 [ 0.  0.  2.  1.  3.  0.  2.  0.  0.  1.  1.  0.  0.  0.  1.  0.  3.  0.
   1.  1.  0.  0.  0.  0.  0.]
 [ 0.  0.  0.  9.  4.  0.  1.  0.  0.  1. 11.  0.  1.  

In [115]:
np.sum(matrix) == len(mapped_novel) - 1

np.True_

In [118]:
print(matrix[0])

[ 0.  2.  0.  1. 14.  0.  3.  0.  0.  0.  6.  0.  2.  0.  2.  3.  1.  1.
  2.  0.  0.  0.  0.  0.  0.]


In [120]:
for i in range(len(matrix)):
    row_sum = sum(matrix[i])

    if row_sum != 0:
        matrix[i] = matrix[i] / row_sum

In [121]:
matrix

array([[0.        , 0.05405405, 0.        , 0.02702703, 0.37837838,
        0.        , 0.08108108, 0.        , 0.        , 0.        ,
        0.16216216, 0.        , 0.05405405, 0.        , 0.05405405,
        0.08108108, 0.02702703, 0.02702703, 0.05405405, 0.        ,
        0.        , 0.        , 0.        , 0.        , 0.        ],
       [0.03921569, 0.03921569, 0.03921569, 0.03921569, 0.11764706,
        0.05882353, 0.07843137, 0.01960784, 0.01960784, 0.        ,
        0.09803922, 0.05882353, 0.05882353, 0.03921569, 0.1372549 ,
        0.05882353, 0.01960784, 0.        , 0.01960784, 0.        ,
        0.        , 0.        , 0.03921569, 0.01960784, 0.        ],
       [0.02469136, 0.        , 0.09876543, 0.0617284 , 0.08641975,
        0.        , 0.11111111, 0.        , 0.01234568, 0.03703704,
        0.11111111, 0.07407407, 0.        , 0.        , 0.16049383,
        0.13580247, 0.        , 0.        , 0.        , 0.04938272,
        0.        , 0.01234568, 0.01234568, 0.

In [123]:
def count_transition_matrix(mat, int_list):
    for i in range(len(int_list) - 1):
        curr_pos = int_list[i]
        next_pos = int_list[i + 1]
    
        row = curr_pos
        col = next_pos
    
        mat[row, col] += 1

    return mat

In [129]:
def prob_matrix(count_mat):
    for i in range(len(count_mat)):
        row_sum = sum(count_mat[i])
    
        if row_sum != 0:
            count_mat[i] = count_mat[i] / row_sum

    return count_mat

In [127]:
def estimate_markov_1(s):  # s in the input text
    # create dictionary mapping of unique char to int
    text_dict = letters2int(s)
    # convert every char to its corresponding int
    text_converted = string2ints(s)

    # initiating empty matrix
    m = np.zeros((len(text_dict), len(text_dict)))

    # fill matrix with counts

    m = count_transition_matrix(m, text_converted)

    # compute probabilities
    m = prob_matrix(m)

    return m  # m is the computed probability matrix

In [130]:
mm1 = estimate_markov_1(novel_clean)

In [132]:
mm1.shape

(25, 25)

In [133]:
mm1.sum(axis=1)

array([1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
       1., 1., 1., 1., 1., 1., 1., 1.])

## Data Probability given a Model

We have now build a model for our text. Unfortunately, it is quite difficult to check whether our computation is correct staring at the model matrix. Since we are nonetheless interested in verifying the correctness of our algorithm, we will adopt a different strategy.

We have estimated our model from a specific piece of text. In statistical terms, we have computed:

$$ P(\theta | D ) $$

where $\theta$ represents the parameters of the model and $D$ the input data.

In this particular case it is quite easy to compute the inverse, that is:

$$ L(D | \theta) $$

the likelihood of a piece of text under the assumption that it was generated by the Markov model.

In computational terms we need to:
* scan the original text;
* for each pair of characters look up the corresponding probability in the model;
* since we assume each pair is an independent event, we can obtain the total probability for the text by multiplying all the intermediate values.

The last step is possible due to the well known probability relation for **independent** events:

$$ P(A,B) = P(A) P(B) $$

There is one last detail we should take care of. The probabilities in our model are going to be quite small. Say $P(A)=0.02$ and $P(B)=0.03$. When we multiply them together, the result is even smaller: $P(A,B) = 0.0006$.

In the long run, this is going to cause numerical problems. The computer is only precise to a certain level; if we get very small values they became essentially indistinguishable from zero.

Luckily, we can use a trick to sidestep this issue: take the logarithm of the probability. By the property of the logarithm, products turn into sums:

$$ \log(\prod_i p_i) = \sum_i \log(p_i) $$

Instead of computing the likelihood directly, we will therefore compute the log-likelihood.

OK, we are now ready to write some code. Implement a function doing the following:
* iterate over all successive pairs in a text;
* look up the corresponding probability in the model;
* take its logarithm;
* return the sum of all logarithms.

<div class="alert alert-warning">
What happens if you have a transition probability of zero? What is its logarithm?
Can you figure a meaningful way to solve this issue?
</div>

In [140]:
def loglik(ints, m):
    total_log_prob = 0

    for i in range(len(ints) - 1):
        curr_state = ints[i]
        next_state = ints[i + 1]

        p = m[curr_state, next_state]
        
        total_log_prob += np.log(p + 1)

    return total_log_prob

In [141]:
loglik(string2ints(novel_clean), mm1)

np.float64(135.94706062105325)

The value of the log-likelihood is again not particularly informative on its own. We need to compute the log-likelihood on a different text to establish a baseline for comparison.

Let's build a random sequence of characters of the same length of the novel fragment we have used. To this end we will use a function of the `numpy` package.

In [142]:
novel_random = np.random.randint(0, len(unique_dict), len(novel_clean))

Now compute the log-likelihood on this new text (you can reuse the function!) and compare it with the previous result.

The Markov model we have generated "explains" better the first (novel) text or the second (random) one?

In [143]:
loglik(novel_random, mm1)

np.float64(53.08996219886385)

## Language Detection

In the last section we learned how to measure the likelihood that a given piece of text derives from a given Markov model. We can now put this knowledge to some practical use by implementing a language classificator.

The idea is to build a software that can automatically detect the language (such as Italian or English) of a string provided by the user.

To do that, we first need to train the software with some samples of each language. For English, we already have our novel.

In [144]:
english_sample = novel_clean

Now go to the Project Gutenberg website and get a sample of French and Italian. Define two more variables, `french_sample` and `italian_sample`, cleaning each text sample in the same way you've done with the english text.

In [ ]:
def clean_sample(s):
    ...

In [145]:
len(novel)

1639

In [151]:
polish_sample = """
Pan Sherlock Holmes zwykł był wstawać późno, o ile nie czuwał przez
całą noc, a zdarzało mu się to nieraz. Otóż owego dnia wstał
wyjątkowo wcześnie. Jadł śniadanie. Stałem przy kominku. Schyliłem
się i podniosłem laskę, którą nasz gość zostawił wczorajszego
wieczoru. Był to kij gruby, z dużą gałką, utoczoną z drzewa; pod
gałką była srebrna obrączka, a na niej napis: „Jakóbowi Mortimer
M. R. C. S. od przyjaciół C. C. H.”, pod spodem zaś data:
„r. 1884”. Taką laskę staroświecką, mocną, zapewniającą
bezpieczeństwo, zwykli nosić starzy lekarze.

-- No, i cóż, Watson? Co pan myślisz o tej lasce? Jakie wyprowadzasz
wnioski? -- zapytał.

Holmes siedział, odwrócony do mnie plecami; widzieć mnie nie mógł,
a ja zachowywałem się tak cichutko, że nie mógł domyślić się, czem
jestem zajęty.

-- Masz pan chyba oczy z tyłu głowy... -- rzekłem.

-- Mam przed sobą srebrny imbryk -- odparł -- ale powiedz mi, Watson,
co myślisz o lasce naszego gościa? Skoro nie zastał nas w domu i nie
wytłómaczył celu swych odwiedzin, ta mimowolna pamiątka nabiera
wielkiego znaczenia. Chciałbym też wiedzieć, jakie pojęcie tworzysz
sobie o tym człowieku?

-- Sądzę -- odparłem, trzymając się metody dociekań mojego
towarzysza -- że doktor Mortimer jest lekarzem średniego wieku,
zażywającym szacunku; dowodzi tego ów upominek pacyentów.

-- Dobrze, wybornie! -- pochwalił mnie Holmes.

"""

In [155]:
polish_sample = clean_string(polish_sample)

In [159]:
polish_sample

'pan sherlock holmes zwykł był wstawać późno o ile nie czuwał przez całą noc a zdarzało mu się to nieraz otóż owego dnia wstał wyjątkowo wcześnie jadł śniadanie stałem przy kominku schyliłem się i podniosłem laskę którą nasz gość zostawił wczorajszego wieczoru był to kij gruby z dużą gałką utoczoną z drzewa pod gałką była srebrna obrączka a na niej napis „jakóbowi mortimer m r c s od przyjaciół c c h pod spodem zaś data „r 1884 taką laskę staroświecką mocną zapewniającą bezpieczeństwo zwykli nosić starzy lekarze no i cóż watson co pan myślisz o tej lasce jakie wyprowadzasz wnioski zapytał holmes siedział odwrócony do mnie plecami widzieć mnie nie mógł a ja zachowywałem się tak cichutko że nie mógł domyślić się czem jestem zajęty masz pan chyba oczy z tyłu głowy rzekłem mam przed sobą srebrny imbryk odparł ale powiedz mi watson co myślisz o lasce naszego gościa skoro nie zastał nas w domu i nie wytłómaczył celu swych odwiedzin ta mimowolna pamiątka nabiera wielkiego znaczenia chciałbym 

<div class="alert alert-info">
Compare the cleaned-up version of the french text with the original. Do you notice any significant difference?
<br><br>
When you've identified it, contact one of the instructors.
</div>

In [160]:
finish_sample = """
Herra Sherlock Holmes, joka tavallisesti nousi hyvin myöhään ylös
aamusin, paitsi niissä kylläkin useissa tapauksissa, jolloin hän oli
valvonut koko yön, istui aamiaisella. Minä seisoin matolla tulisijan
edessä pitäen kädessäni keppiä, jonka eräs edellisenä iltana luonamme
käynyt herra oli unohtanut. Se oli jokseenkin soma ja tukeva, se oli
varustettu sipulinmuotoisella kädensijalla ja näytti oikealta
"tuomarin sauvalta." 'M.R.C.S. [M.R.C.S. Member of the Royal College of
Surgeons = kuninkaallisen kirurgi-kollegion jäsen.] James Mortimerille
ystäviltänsä C. C. H:ssa' oli kaiverrettu tuuman-levyiselle, kädensijan
alapuolella olevalle hopealevylle, sekä vielä vuosiluku 1884. Juuri
sellaista keppiä käyttävät tavallisesti vanhat perhelääkärit -- se
näytti arvokkaalta, vakavalta ja luottamusta herättävältä.

"Kas niin, Watson, mitä johtopäätöksiä tuosta teet?"

Holmes istui selin minuun, enkä ollut sanallakaan ilmottanut mitä
toimitin.

"Kuinka tiesit, mitä minä tein? Luulenpa että sinulla on silmät
niskassakin."

"Ainakin minulla on kiillotettu hopeakannu pöydällä", sanoi hän.
"Mutta sanohan, Watson, mitä saat tietää tuosta kepistä? Koska emme,
ikävä kyllä, saaneet tavata sen omistajaa, emmekä voi aavistaakaan
hänen asiaansa, on tämä hänen käynnistään jäänyt pieni muisto meille
hyvin tärkeä. Muodosta nyt kuva miehestä, kun olet tätä hänen
omistamaansa esinettä niin tarkoin tutkinut."

"Minä luulen", sanoin minä, koettaen seurata toverini menettelytapaa
niin hyvin kuin taisin, "että tohtori Mortimer on vanhanpuoleinen
lääkäri, jolla on melkoinen käyttäjäjoukko, ja arvattavasti pidetään
hänestä, koska hänen tuttavansa ovat tahtoneet antaa hänelle tämän
todistuksen suosiostaan."

"Se on hyvä", sanoi Holmes, "erittäin hyvä."
"""

In [162]:
finnish_sample = clean_string(finish_sample)

finish_sample

'\nHerra Sherlock Holmes, joka tavallisesti nousi hyvin myöhään ylös\naamusin, paitsi niissä kylläkin useissa tapauksissa, jolloin hän oli\nvalvonut koko yön, istui aamiaisella. Minä seisoin matolla tulisijan\nedessä pitäen kädessäni keppiä, jonka eräs edellisenä iltana luonamme\nkäynyt herra oli unohtanut. Se oli jokseenkin soma ja tukeva, se oli\nvarustettu sipulinmuotoisella kädensijalla ja näytti oikealta\n"tuomarin sauvalta." \'M.R.C.S. [M.R.C.S. Member of the Royal College of\nSurgeons = kuninkaallisen kirurgi-kollegion jäsen.] James Mortimerille\nystäviltänsä C. C. H:ssa\' oli kaiverrettu tuuman-levyiselle, kädensijan\nalapuolella olevalle hopealevylle, sekä vielä vuosiluku 1884. Juuri\nsellaista keppiä käyttävät tavallisesti vanhat perhelääkärit -- se\nnäytti arvokkaalta, vakavalta ja luottamusta herättävältä.\n\n"Kas niin, Watson, mitä johtopäätöksiä tuosta teet?"\n\nHolmes istui selin minuun, enkä ollut sanallakaan ilmottanut mitä\ntoimitin.\n\n"Kuinka tiesit, mitä minä tein?

Now build three models, one for each sample.

In [164]:
# english_model = estimate_markov_1(english_sample)
polish_model = estimate_markov_1(polish_sample)
# italian_model = estimate_markov_1(italian_sample)

We are ready to build the classifier. Write a function that receives a single string input. It has to compute the log-likelihood of that string according to the different models and then pick the best one. The function return value should be the name of the guessed language.

In [ ]:
def guess_language(s):
    ...

In [ ]:
guess_language("ecco un messaggio scritto in italiano")

In [ ]:
guess_language("a sample message written in plain english gets a good classification")

In [ ]:
guess_language('''
Allons enfants de la Patrie
Le jour de gloire est arrivé!
Contre nous de la tyrannie
''')

## Performance Evaluation

<div class="alert alert-info">
This section is optional.
</div>

No classifier is perfect. Here we should put our function `guess_language` to the test and check in which conditions it works and in which it provides a wrong result.
There are two choices having a primary impact of performances: the size of the training dataset (text) and that of the message to be classified.

Here we will concentrate on the second, that is the message length. Write a function `extract_with_length` to extract a random selection out of a string, given a desired length.

For instance:

```python
extract_with_length('a simple message', 6)
```

should return, each time it is called, a (different) random substring of length 6.

Examples:

```python
>>> extract_with_length('a simple message', 6)
"a simp"
>>> extract_with_length('a simple message', 6)
"messag"
```

<div class="alert alert-info">
Import the ```random``` library. You can then use the ```random.randint``` function to generate random integer numbers. Use the Jupyter documentation to check how the function works.
</div>

In [ ]:
import random

In [ ]:
def extract_with_length(s, n):
    ...

What happens if I run the following code:

```python
extract_with_length('short', 20)
```

How can I avoid this problem, while improving the error message?

We can now implement a Monte Carlo simulation to evaluate the performance of our classifier with respect to the length of the input message.

* choose a length `l` (for instance, `l=10`) and extract 1000 random strings with `extract_with_length`;
* run `guess_language` on those strings and record whether the classifier was right or wrong;
* calculate the performance as the ratio between correct and total classifications;
* repeat the above procedure for all lengths `l` between 5 and 40.

Prepare a scatter plot displaying the value of `l` on the $x$ axis and the performance on the $y$ axis.

Do you see any trend? Can you explain it?

In [ ]:
import matplotlib.pylab as plt
%matplotlib inline

In [ ]:
def performance_monte_carlo(template, expected_language):
    ... 
    plt.plot(X, Y)

In [ ]:
performance_monte_carlo(clean_sample('''
Va’, pensiero, sull'ali dorate;
va’, ti posa sui clivi, sui colli,
ove olezzano tepide e molli
l'aure dolci del suolo natal!

Del Giordano le rive saluta,
di Sionne le torri atterrate...
Oh, mia patria sì bella e perduta!
Oh, membranza sì cara e fatal!

Arpa d'or dei fatidici vati,
perché muta dal salice pendi?
Le memorie nel petto raccendi,
ci favella del tempo che fu!

O simile di Solima ai fati
traggi un suono di crudo lamento,
o t'ispiri il Signore un concento
che ne infonda al patire virtù!
'''), 'italian')

## Appendix A

In [ ]:
import unicodedata
def sa(s):
    return ''.join(c for c in unicodedata.normalize('NFD', s)
                   if unicodedata.category(c) != 'Mn')